# 01 — TF-IDF Analizi ve Kelime Bulutları

**Amaç:** Toplanan App Store + Google Play yorumlarında pozitif ve negatif sınıflarda öne çıkan kelimeleri TF-IDF ile bulup kelime bulutu olarak görselleştirmek.

**Etiketleme:** `rating` 1-2 → negatif, 3 → nötr, 4-5 → pozitif.

**Çıktılar:** `visuals/` altında PNG'ler ve `data/processed/` altında top-kelime CSV'leri.

In [ ]:
import os
import re
import string
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from wordcloud import WordCloud
import nltk

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110

PROJECT_ROOT = Path('..').resolve()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
VISUALS_DIR = PROJECT_ROOT / 'visuals'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
VISUALS_DIR.mkdir(parents=True, exist_ok=True)

print('Proje kökü:', PROJECT_ROOT)

In [ ]:
try:
    from nltk.corpus import stopwords
    TR_STOPWORDS = set(stopwords.words('turkish'))
except LookupError:
    nltk.download('stopwords')
    from nltk.corpus import stopwords
    TR_STOPWORDS = set(stopwords.words('turkish'))

EXTRA_STOPWORDS = {
    'bir', 'çok', 'daha', 'şey', 'şu', 'şöyle', 'şimdi', 'bende', 'bana',
    'bende', 'sonra', 'önce', 'kadar', 'gibi', 'oluyor', 'oldu', 'olmuş',
    'var', 'yok', 'evet', 'hayır', 'ama', 'fakat', 'tam', 'hep', 'hiç',
    'uygulama', 'uygulamayı', 'uygulamanın', 'uygulamada',
    'telefon', 'telefonum', 'telefonumda',
    'app', 'video', 'videolar', 'youtube', 'instagram', 'whatsapp', 'tiktok',
    'için', 'ile', 'ki', 'mi', 'mı', 'mu', 'mü', 'da', 'de', 'ta', 'te'
}
STOPWORDS = TR_STOPWORDS | EXTRA_STOPWORDS
print(f'Toplam stopword: {len(STOPWORDS)}')

In [ ]:
app_store = pd.read_csv(RAW_DIR / 'app_store_reviews.csv')
google_play = pd.read_csv(RAW_DIR / 'google_play_reviews.csv')
df = pd.concat([app_store, google_play], ignore_index=True)

df['text'] = df['text'].fillna('').astype(str)
df['title'] = df['title'].fillna('').astype(str)
df['full_text'] = (df['title'] + ' ' + df['text']).str.strip()
df = df[df['full_text'].str.len() > 0].copy()
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
df = df.dropna(subset=['rating'])
df['rating'] = df['rating'].astype(int)

print(f'Toplam yorum: {len(df):,}')
print(df['platform'].value_counts())

In [ ]:
def label_sentiment(rating: int) -> str:
    if rating <= 2:
        return 'negatif'
    if rating == 3:
        return 'nötr'
    return 'pozitif'

df['sentiment'] = df['rating'].apply(label_sentiment)
print(df['sentiment'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

order_r = sorted(df['rating'].unique())
sns.countplot(data=df, x='rating', order=order_r, palette='viridis', ax=axes[0])
axes[0].set_title('Puan (rating) dağılımı')
axes[0].set_xlabel('Yıldız')
axes[0].set_ylabel('Yorum sayısı')

order_s = ['negatif', 'nötr', 'pozitif']
sns.countplot(data=df, x='sentiment', order=order_s,
              palette=['#d62728', '#7f7f7f', '#2ca02c'], ax=axes[1])
axes[1].set_title('Sentiment dağılımı')
axes[1].set_xlabel('Sınıf')
axes[1].set_ylabel('Yorum sayısı')

plt.tight_layout()
plt.savefig(VISUALS_DIR / 'sentiment_distribution.png', bbox_inches='tight')
plt.show()

## Metin temizleme

Küçük harfe çevir, URL/sayı/noktalama at, stopword'leri ve 2 karakterden kısa token'ları sil.

In [ ]:
URL_RE = re.compile(r'https?://\S+|www\.\S+')
NON_WORD_RE = re.compile(r'[^a-zçğıöşü\s]+', flags=re.IGNORECASE)
MULTI_SPACE_RE = re.compile(r'\s+')

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    t = text.lower()
    t = t.replace('İ', 'i').replace('I', 'ı')
    t = URL_RE.sub(' ', t)
    t = NON_WORD_RE.sub(' ', t)
    t = MULTI_SPACE_RE.sub(' ', t).strip()
    tokens = [w for w in t.split() if len(w) > 2 and w not in STOPWORDS]
    return ' '.join(tokens)

df['clean'] = df['full_text'].apply(clean_text)
df = df[df['clean'].str.len() > 0].copy()
print('Temizleme sonrası örnek:')
df[['rating', 'sentiment', 'full_text', 'clean']].head(5)

## TF-IDF analizi

Her yorum bir doküman. `TfidfVectorizer` tüm korpus üzerinde fit edilir, sonra her sentiment sınıfı için sütun (kelime) ortalama TF-IDF skoru alınır. Yüksek ortalama → o sınıfa özgü kelime.

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 1),
    min_df=5,
    max_df=0.85,
)
X = vectorizer.fit_transform(df['clean'])
vocab = np.array(vectorizer.get_feature_names_out())
print('Sözlük boyutu:', len(vocab))
print('Matris şekli:', X.shape)

In [ ]:
def class_mean_tfidf(sentiment_label: str) -> pd.Series:
    mask = (df['sentiment'] == sentiment_label).values
    if mask.sum() == 0:
        return pd.Series(dtype=float)
    means = np.asarray(X[mask].mean(axis=0)).ravel()
    return pd.Series(means, index=vocab).sort_values(ascending=False)

TOP_N = 30
top_pos = class_mean_tfidf('pozitif').head(TOP_N)
top_neg = class_mean_tfidf('negatif').head(TOP_N)
top_neu = class_mean_tfidf('nötr').head(TOP_N)

top_pos.to_csv(PROCESSED_DIR / 'top_words_positive.csv', header=['tfidf'])
top_neg.to_csv(PROCESSED_DIR / 'top_words_negative.csv', header=['tfidf'])
top_neu.to_csv(PROCESSED_DIR / 'top_words_neutral.csv', header=['tfidf'])

comparison = pd.concat(
    [top_pos.head(20).rename('pozitif').reset_index(drop=True),
     pd.Series(top_pos.head(20).index, name='pozitif_kelime'),
     top_neg.head(20).rename('negatif').reset_index(drop=True),
     pd.Series(top_neg.head(20).index, name='negatif_kelime')],
    axis=1,
)
comparison[['pozitif_kelime', 'pozitif', 'negatif_kelime', 'negatif']]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 8))

top_pos.head(20).iloc[::-1].plot(kind='barh', color='#2ca02c', ax=axes[0])
axes[0].set_title('Pozitif yorumlarda öne çıkan kelimeler (TF-IDF)')
axes[0].set_xlabel('Ortalama TF-IDF')

top_neg.head(20).iloc[::-1].plot(kind='barh', color='#d62728', ax=axes[1])
axes[1].set_title('Negatif yorumlarda öne çıkan kelimeler (TF-IDF)')
axes[1].set_xlabel('Ortalama TF-IDF')

plt.tight_layout()
plt.savefig(VISUALS_DIR / 'top_words_pos_neg.png', bbox_inches='tight')
plt.show()

## Kelime bulutları

Pozitif ve negatif sınıflar için ayrı bulut. Boyutlar TF-IDF ortalama skoruna göre.

In [ ]:
def make_wordcloud(freqs: pd.Series, colormap: str, title: str, filename: str):
    wc = WordCloud(
        width=1400, height=700,
        background_color='white',
        colormap=colormap,
        prefer_horizontal=0.9,
        random_state=42,
    ).generate_from_frequencies(freqs.to_dict())
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(title, fontsize=14)
    plt.tight_layout()
    plt.savefig(VISUALS_DIR / filename, bbox_inches='tight', dpi=150)
    plt.show()

pos_freqs = class_mean_tfidf('pozitif').head(150)
neg_freqs = class_mean_tfidf('negatif').head(150)

make_wordcloud(pos_freqs, 'Greens', 'Pozitif yorumlarda öne çıkan kelimeler', 'wordcloud_positive.png')
make_wordcloud(neg_freqs, 'Reds', 'Negatif yorumlarda öne çıkan kelimeler', 'wordcloud_negative.png')

## Çıktılar

- `visuals/sentiment_distribution.png` — puan ve sentiment dağılımı
- `visuals/top_words_pos_neg.png` — pozitif/negatif top 20 kelime bar grafiği
- `visuals/wordcloud_positive.png` — pozitif kelime bulutu
- `visuals/wordcloud_negative.png` — negatif kelime bulutu
- `data/processed/top_words_positive.csv` — pozitif sınıfta top 30 kelime + TF-IDF skorları
- `data/processed/top_words_negative.csv` — negatif sınıfta top 30 kelime + TF-IDF skorları
- `data/processed/top_words_neutral.csv` — nötr sınıfta top 30 kelime + TF-IDF skorları

**Sonraki adım:** BERT (`nlptown/bert-base-multilingual-uncased-sentiment`) ile model kurulumu — `notebooks/02_bert_sentiment.ipynb`.